In [3]:
!pip install -q -U google-genai

In [18]:
from google import genai
from google.colab import userdata
from google.genai import types
import numpy as np
import json

client=genai.Client(api_key=userdata.get("GEMINI_API_KEYS"))
MODEL="gemini-3.5-flash-lite"
EMB_MODEL="gemini-embedding-001"
EMB_DIM=768

In [19]:
notes=input("Enter notes: ")
prompt=f"""
You are Senior Notes maker.
So you have given paragraph of notes on a topic in {notes}.
So return List containing each point as 1element [point1(string),point2,point3.........]
"""
response=client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0.2,
        max_output_tokens=2000,
        system_instruction="You are AI NOTES MAKER,reply starting with HI,I AM YOU ARE NOTES MAKING AGENT",
        thinking_config=types.ThinkingConfig(thinking_level='high')
    )
)
note_points=response.text

def embed(text):
  response=client.models.embed_content(
      model=EMB_MODEL,
      contents=text,
      config=types.EmbedContentConfig(
          output_dimensionality=EMB_DIM
      )
  )
  return response.embeddings[0].values

def cosine_sim(a,b):
  a=np.array(a)
  b=np.array(b)

  return (np.dot(a,b))/(np.linalg.norm(a)*np.linalg.norm(b))

embeddings=np.array([
    embed(point)
    for point in note_points
])

user_question=input("Enter point you need to learn")
user_embed=embed(user_question)
similarity=[]
for i,embedding in enumerate(embeddings):
  similarity.append((i,(cosine_sim(embeddings[i],user_embed))))

best_index,best_score=max(similarity,key=lambda x:x[1])
point=note_points[best_index]
prompt_point=f"""
You are given with notepoint :{point}
based on point the explain the point in detail.
"""
response_point=client.models.generate_content(
    model=MODEL,
    contents=prompt_point,
    config=types.GenerateContentConfig(
        temperature=0.2,
        max_output_tokens=1000,
        system_instruction="You are AI NOTES MAKER,reply starting with HI,I AM YOU ARE NOTES MAKING AGENT",
        thinking_config=types.ThinkingConfig(thinking_level='high')
    )
)
print(response_point.text)

Enter notes: Artificial Intelligence (AI) is a branch of computer science that focuses on creating machines and software capable of performing tasks that normally require human intelligence. These tasks include understanding language, recognizing images, solving problems, making decisions, and learning from data. AI systems use techniques such as machine learning, deep learning, natural language processing, and computer vision. Machine learning allows computers to identify patterns in data and make predictions without being explicitly programmed for every situation. Deep learning uses neural networks with multiple layers to process complex data such as images, audio, and text. Natural Language Processing (NLP) enables computers to understand and generate human language. AI is widely used in applications such as chatbots, recommendation systems, fraud detection, autonomous vehicles, healthcare, and virtual assistants.


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 57.54627942s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-embedding-1.0', 'locat